In [1]:
import os
import pandas as pd
import numpy as np
from scipy import stats
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

load_dotenv("../.env")
url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)
engine = create_engine(url)

date_cols = ["order_purchase_timestamp", "order_delivered_customer_date",
             "order_estimated_delivery_date"]
orders = pd.read_csv("../data/cleaned/orders_features.csv", parse_dates=date_cols)
reviews = pd.read_sql("SELECT * FROM order_reviews", engine)
payments = pd.read_sql("SELECT * FROM order_payments", engine)
items = pd.read_sql("SELECT * FROM order_items", engine)
products = pd.read_sql("SELECT * FROM products", engine)
translation = pd.read_sql("SELECT * FROM product_category_translation", engine)
print("Ready!")

Ready!


In [2]:
delivered = orders[orders["order_status"] == "delivered"]
data = delivered.merge(reviews[["order_id", "review_score"]], on="order_id")

late = data.loc[data["is_delayed"] == True, "review_score"]
on_time = data.loc[data["is_delayed"] == False, "review_score"]

print("Delayed:", len(late), "mean", round(late.mean(), 2))
print("On time:", len(on_time), "mean", round(on_time.mean(), 2))

# Assumption check: do both groups have similar spread?
print("Levene p-value:", stats.levene(late, on_time).pvalue)

# Welch's t-test (safe when spreads differ), one-sided: late < on_time
t_stat, p_value = stats.ttest_ind(late, on_time, equal_var=False, alternative="less")
print("t-statistic:", round(t_stat, 2))
print("p-value:", p_value)
print("Decision:", "Reject H0" if p_value < 0.05 else "Fail to reject H0")

Delayed: 6361 mean 2.27
On time: 89248 mean 4.29
Levene p-value: 4.4001401791529906e-293
t-statistic: -100.78
p-value: 0.0
Decision: Reject H0


In [3]:
# Category per order = category of the first item (simple and common approach)
first_item = items.sort_values("order_item_id").drop_duplicates("order_id")
cat = first_item.merge(products[["product_id", "product_category_name"]], on="product_id")
cat = cat.merge(translation, on="product_category_name", how="left")
cat["category"] = cat["product_category_name_english"].fillna(cat["product_category_name"])

anova_data = cat.merge(orders[["order_id", "total_order_value"]], on="order_id").dropna(subset=["total_order_value"])

# Keep categories with at least 100 orders, so each group is reliable
counts = anova_data["category"].value_counts()
anova_data = anova_data[anova_data["category"].isin(counts[counts >= 100].index)]
print("Categories used:", anova_data["category"].nunique(), "| Orders:", len(anova_data))

groups = [g["total_order_value"].values for _, g in anova_data.groupby("category")]

# Assumption checks
print("Levene p-value:", stats.levene(*groups).pvalue)

f_stat, p_anova = stats.f_oneway(*groups)
print("ANOVA F:", round(f_stat, 2), "| p-value:", p_anova)

# Kruskal-Wallis: backup test, because order values are heavily skewed
h_stat, p_kw = stats.kruskal(*groups)
print("Kruskal-Wallis p-value:", p_kw)

Categories used: 52 | Orders: 97814
Levene p-value: 0.0
ANOVA F: 201.44 | p-value: 0.0
Kruskal-Wallis p-value: 0.0


In [4]:
cat_summary = (anova_data.groupby("category")["total_order_value"]
               .agg(["mean", "median", "count"])
               .sort_values("mean", ascending=False))
print(cat_summary.head(5))
print(cat_summary.tail(5))

                                   mean    median  count
category                                                
computers                   1285.524111  1250.810    180
home_appliances_2            530.977682   287.270    233
agro_industry_and_commerce   430.626758   410.615    182
musical_instruments          336.648173   145.945    624
small_appliances             329.053541   128.620    627
                         mean  median  count
category                                    
christmas_supplies  94.151641  75.815    128
books_technical     90.730039  69.040    259
food_drink          87.574420  60.240    224
electronics         81.955858  37.860   2540
food                81.550225  71.245    444


In [5]:
# One payment type per order: the one with the largest payment value
main_pay = (payments.sort_values("payment_value", ascending=False)
            .drop_duplicates("order_id")[["order_id", "payment_type"]])
chi_data = main_pay.merge(orders[["order_id", "order_status"]], on="order_id")

# Merge tiny statuses so every cell has enough observations
chi_data["status_group"] = chi_data["order_status"].where(
    chi_data["order_status"].isin(["delivered", "shipped", "canceled", "unavailable"]),
    "other")

table = pd.crosstab(chi_data["payment_type"], chi_data["status_group"])
print(table)

chi2, p_chi, dof, expected = stats.chi2_contingency(table)
print("\nChi-square:", round(chi2, 2), "| dof:", dof, "| p-value:", p_chi)
print("Smallest expected count:", round(expected.min(), 2))

# Effect size: Cramer's V (0 = no association, 1 = perfect)
n = table.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
print("Cramer's V:", round(cramers_v, 3))

status_group  canceled  delivered  other  shipped  unavailable
payment_type                                                  
boleto              95      19191    139      209          150
credit_card        432      72825    455      836          427
debit_card           7       1484      8       22            6
voucher             88       2977     20       40           26

Chi-square: 267.02 | dof: 12 | p-value: 3.823288708992219e-50
Smallest expected count: 9.35
Cramer's V: 0.03
